# ChuckleNet: Complete Scale Pipeline (v3 - FAST)

## Key Fixes:
- **Install soundfile FIRST** → no more audioread fallback (10x faster)
- **Load audio ONCE** → resample to both sr in memory (not twice from disk)
- **Batch utterance processing** → process multiple utterances per file
- **Checkpoint every 20 videos** → faster saves

**Data:** `chuckle_net_1000/` (621 audio, 628 VTT)
**Runtime:** ~20-30 min (was 5-14 hours)


In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive')

# CRITICAL: Install soundfile FIRST - avoids audioread fallback (10x slower!)
!pip install -q soundfile librosa numpy pandas scikit-learn torch transformers tqdm

import numpy as np
import glob
from tqdm import tqdm
import torch
import soundfile as sf
import librosa

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Paths
BASE = '/content/drive/MyDrive/chuckle_net_1000'
AUDIO_DIR = f'{BASE}/audio'
VTT_DIR = f'{BASE}/vtt'
OUTPUT = '/content/drive/MyDrive/chuckle_net_output'
os.makedirs(OUTPUT, exist_ok=True)

audio_files = sorted(glob.glob(f'{AUDIO_DIR}/*.m4a'))
print(f'Audio: {len(audio_files)} files')
print(f'Output: {OUTPUT}')

In [ ]:
# === CHECK FOR EXISTING CHECKPOINTS ===
CHECKPOINT_FILE = f'{OUTPUT}/extraction_checkpoint.npz'
CHECKPOINT_IDX = f'{OUTPUT}/processed_idx.txt'

processed_idx = set()
if os.path.exists(CHECKPOINT_IDX):
    with open(CHECKPOINT_IDX, 'r') as f:
        processed_idx = set(f.read().splitlines())
    print(f'Resuming from {len(processed_idx)} already processed files')

def save_checkpoint(features, labels, vids, utterance_count):
    """Save checkpoint with all data"""
    np.savez_compressed(CHECKPOINT_FILE,
                        features=features,
                        labels=labels,
                        vids=np.array(vids),
                        utterance_counts=utterance_count)
    with open(CHECKPOINT_IDX, 'w') as f:
        f.write('\n'.join(sorted(processed_idx)))
    print(f'✓ Checkpoint saved: {len(features)} samples, {utterance_count} utterances')

def load_checkpoint():
    """Load checkpoint data"""
    if os.path.exists(CHECKPOINT_FILE):
        data = np.load(CHECKPOINT_FILE, allow_pickle=True)
        return data['features'], data['labels'], data['vids']
    return None, None, None

In [ ]:
# === UTTERANCE-LEVEL VTT PARSING ===

def parse_vtt_for_laughter(vtt_path):
    """Extract utterance boundaries and laughter labels from VTT."""
    try:
        with open(vtt_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        utterances = []
        has_laughter = False
        
        for line in content.split('\n'):
            line = line.strip()
            if '-->' in line:
                if utterances:
                    utterances[-1]['has_laughter'] = has_laughter
                
                parts = line.split('-->')
                start = float(parts[0].strip().replace(',', '.'))
                end = float(parts[1].strip().split()[0].replace(',', '.'))
                
                utterances.append({
                    'start': start,
                    'end': end,
                    'text': [],
                    'has_laughter': False
                })
                has_laughter = False
                
            elif '[laughter]' in line.lower():
                has_laughter = True
            elif utterances and not line.startswith('<'):
                utterances[-1]['text'].append(line)
        
        if utterances:
            utterances[-1]['has_laughter'] = has_laughter
        
        return utterances
    except:
        return None

In [ ]:
# === FAST AUDIO LOADING (load once, resample in memory) ===

def load_audio(audio_path, sr_target=(16000, 22050)):
    """Load audio once and resample to multiple sample rates in memory.
    
    Returns dict of sr -> waveform
    """
    try:
        # Try soundfile first (fast, no fallback)
        try:
            y_orig, sr_orig = sf.read(audio_path)
            y_orig = y_orig.astype(np.float32)
            if y_orig.ndim > 1:
                y_orig = y_orig.mean(axis=1)  # stereo -> mono
        except:
            # Fallback to librosa with soundfile installed
            y_orig, sr_orig = librosa.load(audio_path, sr=None, mono=True)
        
        result = {}
        for sr in sr_target:
            if sr_orig == sr:
                result[sr] = y_orig
            else:
                result[sr] = librosa.resample(y_orig, orig_sr=sr_orig, target_sr=sr)
        
        return result
    except Exception as e:
        return None

In [ ]:
# === PROSODY EXTRACTION PER UTTERANCE (21-dim) ===

def extract_prosody_segment(y, sr, start_s, end_s, hop_length=512):
    """Extract 21-dim prosody features for ONE segment."""
    start_sample = int(start_s * sr)
    end_sample = int(end_s * sr)
    y_seg = y[start_sample:end_sample]
    
    if len(y_seg) < sr * 0.1:  # <100ms
        return None
    
    features = []
    
    # RMS energy (3)
    rms = librosa.feature.rms(y=y_seg, hop_length=hop_length)[0]
    features.extend([np.mean(rms), np.std(rms), np.max(rms)])
    
    # F0 using pyin (3)
    f0, voiced, prob = librosa.pyin(y_seg, fmin=50, fmax=500, sr=sr, hop_length=hop_length)
    f0 = np.nan_to_num(f0, nan=0)
    f0_valid = f0[f0 > 0]
    if len(f0_valid) > 0:
        features.extend([np.mean(f0_valid), np.std(f0_valid), np.max(f0_valid) - np.min(f0_valid)])
    else:
        features.extend([0, 0, 0])
    
    # ZCR (2)
    zcr = librosa.feature.zero_crossing_rate(y_seg, hop_length=hop_length)[0]
    features.extend([np.mean(zcr), np.std(zcr)])
    
    # Spectral centroid (2)
    sc = librosa.feature.spectral_centroid(y=y_seg, sr=sr, hop_length=hop_length)[0]
    features.extend([np.mean(sc), np.std(sc)])
    
    # Spectral bandwidth (2)
    sb = librosa.feature.spectral_bandwidth(y=y_seg, sr=sr, hop_length=hop_length)[0]
    features.extend([np.mean(sb), np.std(sb)])
    
    # Spectral rolloff (2)
    rolloff = librosa.feature.spectral_rolloff(y=y_seg, sr=sr, hop_length=hop_length)[0]
    features.extend([np.mean(rolloff), np.std(rolloff)])
    
    # MFCCs 1-13 (13)
    mfcc = librosa.feature.mfcc(y=y_seg, sr=sr, n_mfcc=13, hop_length=hop_length)
    for i in range(13):
        features.append(np.mean(mfcc[i]))
    
    return np.array(features, dtype=np.float32)

In [ ]:
# === WAVLM EXTRACTION PER UTTERANCE (768-dim) ===

from transformers import Wav2Vec2Model

print('Loading WavLM...')
wavlm = Wav2Vec2Model.from_pretrained('microsoft/wavlm-base')
wavlm.to(DEVICE)
wavlm.eval()
print(f'WavLM loaded on {DEVICE}')

def extract_wavlm_segment(y_16k, start_s, end_s):
    """Extract 768-dim WavLM embedding for ONE segment."""
    start_sample = int(start_s * 16000)
    end_sample = int(end_s * 16000)
    y_seg = y_16k[start_sample:end_sample]
    
    if len(y_seg) < 1600:  # <100ms
        return None
    
    with torch.no_grad():
        inputs = torch.FloatTensor(y_seg).unsqueeze(0).to(DEVICE)
        out = wavlm(inputs)
        emb = out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    
    return emb.astype(np.float32)

In [ ]:
# === MAIN EXTRACTION LOOP (utterance-level, checkpoint every 20) ===

def process_all_videos(audio_files, start_idx=0):
    """Process all videos at UTTERANCE level."""
    
    all_features = []  # [prosody_21 + wavlm_768 = 789]
    all_labels = []
    all_vids = []
    
    checkpoint_interval = 20
    
    for i, af in enumerate(tqdm(audio_files[start_idx:], desc='Processing')):
        vid = os.path.basename(af).replace('.m4a', '')
        
        # Find VTT
        vtt_patterns = glob.glob(f'{VTT_DIR}/{vid}.*.vtt')
        utterances = None
        if vtt_patterns:
            utterances = parse_vtt_for_laughter(vtt_patterns[0])
        
        if not utterances:
            continue
        
        # Load audio ONCE, resample to both sr in memory
        audio_data = load_audio(af, sr_target=(16000, 22050))
        if audio_data is None:
            continue
        
        y_16k = audio_data[16000]
        y_22k = audio_data[22050]
        
        # Process each utterance
        for utt in utterances:
            prosody = extract_prosody_segment(y_22k, 22050, utt['start'], utt['end'])
            wavlm_emb = extract_wavlm_segment(y_16k, utt['start'], utt['end'])
            
            if prosody is not None and wavlm_emb is not None:
                # Concatenate: 768 + 21 = 789
                feat = np.concatenate([wavlm_emb, prosody])
                all_features.append(feat)
                all_labels.append(1 if utt['has_laughter'] else 0)
                all_vids.append(vid)
        
        processed_idx.add(vid)
        
        # Checkpoint every 20 videos
        if (i + 1) % checkpoint_interval == 0:
            save_checkpoint(
                np.array(all_features),
                np.array(all_labels),
                all_vids,
                len(all_features)
            )
    
    return np.array(all_features), np.array(all_labels), np.array(all_vids)

# Load checkpoint if exists
existing_features, existing_labels, existing_vids = load_checkpoint()

# Determine starting index
start_idx = 0
if processed_idx:
    for i, af in enumerate(audio_files):
        vid = os.path.basename(af).replace('.m4a', '')
        if vid not in processed_idx:
            start_idx = i
            break
    else:
        print(f'Already processed all {len(audio_files)} files!')
        
print(f'Starting from index {start_idx}/{len(audio_files)}')

# Process
X, y, vids = process_all_videos(audio_files, start_idx)
print(f'\nExtracted: {len(X)} utterance-level samples')
print(f'Features: {X.shape}')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')

In [ ]:
# === SAVE FINAL FEATURES ===

np.savez_compressed(f'{OUTPUT}/utterance_features.npz',
                    features=X, labels=y, vids=vids)

print(f'Saved: {OUTPUT}/utterance_features.npz')
print(f'Shape: {X.shape} (samples x features)')
print(f'Positive rate: {100*y.mean():.1f}%')

In [ ]:
# === FUSION MLP TRAINING (GPU) ===

import torch
import torch.nn as nn
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

# Load data
data = np.load(f'{OUTPUT}/utterance_features.npz')
X = data['features']
y = data['labels']
vids = data['vids']

print(f'Data: {len(y)} samples, {len(set(vids))} videos')
print(f'Features: {X.shape}')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')

# Standardize
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

# Fusion MLP
class FusionMLP(nn.Module):
    def __init__(self, dim=789, hidden=[512, 256, 64]):
        super().__init__()
        self.bn0 = nn.BatchNorm1d(dim)
        layers = []
        prev = dim
        for h in hidden:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(0.3)
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(self.bn0(x)).squeeze(-1)

# 5-fold video-level CV
gkf = GroupKFold(n_splits=5)
f1s, precs, recs = [], [], []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_s, y, vids)):
    X_tr = torch.FloatTensor(X_s[tr_idx])
    y_tr = torch.FloatTensor(y[tr_idx])
    X_te = torch.FloatTensor(X_s[te_idx])
    y_te = y[te_idx]
    
    model = FusionMLP(dim=X_s.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)
    
    pos_weight = torch.tensor([(1-y_tr.mean())/max(0.01, y_tr.mean())]).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    for epoch in range(100):
        model.train()
        for i in range(0, len(X_tr), 32):
            batch_x = X_tr[i:i+32].to(DEVICE)
            batch_y = y_tr[i:i+32].to(DEVICE)
            
            opt.zero_grad()
            loss = loss_fn(model(batch_x), batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
    
    model.eval()
    with torch.no_grad():
        preds = torch.sigmoid(model(X_te.to(DEVICE))).cpu().numpy()
        pred_binary = (preds > 0.5).astype(int)
        
        f1 = f1_score(y_te, pred_binary, zero_division=0)
        prec = precision_score(y_te, pred_binary, zero_division=0)
        rec = recall_score(y_te, pred_binary, zero_division=0)
        
        f1s.append(f1)
        precs.append(prec)
        recs.append(rec)
    
    print(f'Fold {fold+1}: F1={f1:.4f}, P={prec:.4f}, R={rec:.4f}')

print(f'\n=== RESULTS ===')
print(f'Mean F1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
print(f'Mean Precision: {np.mean(precs):.4f}')
print(f'Mean Recall: {np.mean(recs):.4f}')

In [ ]:
# === SAVE FINAL MODEL ===

torch.save({
    'model_state_dict': model.state_dict(),
    'scaler': scaler,
    'f1_mean': np.mean(f1s),
    'f1_std': np.std(f1s)
}, f'{OUTPUT}/fusion_model_final.pt')

print(f'\nModel saved: {OUTPUT}/fusion_model_final.pt')
print(f'Final F1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')